In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

In [ ]:
import os
import requests
import zipfile
import scanpy as sc
import pandas as pd
import glob, tarfile


In [ ]:
from metab_processing.metab_travlr_config import PROJECT_DATA_DIR, DATA_DIR as METAB_DATA_DIR

XENIUM_DATA_DIR = PROJECT_DATA_DIR

In [5]:
def download_file(url, dest_dir, timeout=60):
    """Download url into dest_dir, resuming a partial file if present.
    
    Skips download entirely if the file is already fully downloaded.
    """
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(url))
    headers = {"User-Agent": "Mozilla/5.0"}
    
    # Check remote file size using a HEAD request
    try:
        with requests.head(url, headers=headers, timeout=timeout) as head_res:
            head_res.raise_for_status()
            remote_size = int(head_res.headers.get("Content-Length", 0))
    except requests.RequestException:
        # Fallback if HEAD request is blocked or unsupported by the server
        remote_size = None

    pos = os.path.getsize(dest) if os.path.exists(dest) else 0
    
    # Skip download if local size matches or exceeds remote size
    if remote_size and pos >= remote_size:
        print(f"File already fully downloaded: {dest}")
        return dest

    if pos:
        headers["Range"] = f"bytes={pos}-"
        
    with requests.get(url, headers=headers, stream=True, timeout=timeout) as r:
        # If server returns 416, it usually means we already have the full file
        if r.status_code == 416:
            print(f"File appears complete (Server returned 416). Skipping: {dest}")
            return dest
            
        r.raise_for_status()
        
        # If server doesn't support partial content (returns 200 instead of 206), reset position
        mode = "ab" if (pos and r.status_code == 206) else "wb"
        
        with open(dest, mode) as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
                
    print("Saved to", dest)
    return dest


def unzip_file(zip_path, dest_dir):
    """Extract zip_path into dest_dir. Skips extraction if already unzipped."""
    os.makedirs(dest_dir, exist_ok=True)
    
    # Hidden marker file to guarantee complete extraction
    completion_marker = os.path.join(dest_dir, ".unzip_complete")
    
    if os.path.exists(completion_marker):
        print(f"Directory already extracted: {dest_dir}. Skipping unzip.")
        return dest_dir
        
    print(f"Extracting {zip_path} to {dest_dir}...")
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest_dir)
        
    # Create the marker file after successful extraction
    with open(completion_marker, "w") as f:
        f.write("extraction_complete")
        
    print("Extracted to", dest_dir)
    return dest_dir

def load_and_save_adata(raw, save_to_path=None, overwrite=False):
    """Loads Xenium h5 and parquet metadata into an AnnData object, with overwrite controls."""
    
    # If file exists and overwrite is False, read the saved object and skip processing
    if save_to_path and os.path.exists(save_to_path) and not overwrite:
        print(f"File already exists at {save_to_path}. Loading existing object (overwrite=False).")
        adata = sc.read_h5ad(save_to_path)
        print(adata)
        return adata

    # Otherwise, execute the parsing pipeline
    # expression matrix (cells × genes)
    adata = sc.read_10x_h5(os.path.join(raw, 'cell_feature_matrix.h5'))
    adata.var_names_make_unique()
    
    # cell metadata + coordinates, aligned on cell_id
    cells = pd.read_parquet(os.path.join(raw, 'cells.parquet')).set_index('cell_id')
    adata.obs = adata.obs.join(cells)
    adata.obsm['spatial'] = adata.obs[['x_centroid', 'y_centroid']].to_numpy()
    
    if save_to_path is not None:
        adata.write_h5ad(save_to_path)
        print(f"Saved AnnData object to {save_to_path}")
        
    print(adata)
    return adata

# FFPE Human Ovarian Cancer:
https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-ovarian-cancer

In [26]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'FFPE_Human_Ovarian_Cancer'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

File already fully downloaded: /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/FFPE_Human_Ovarian_Cancer/Xenium_Prime_Ovarian_Cancer_FFPE_XRrun_outs.zip
Directory already extracted: /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/FFPE_Human_Ovarian_Cancer/Raw_Data. Skipping unzip.
File already exists at /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/FFPE_Human_Ovarian_Cancer/adata.h5ad. Loading existing object (overwrite=False).
AnnData object with n_obs × n_vars = 407124 × 5101
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


# Fresh Frozen Human Ovarian Adenocarcinoma
http://10xgenomics.com/datasets/xenium-prime-fresh-frozen-human-ovary

In [4]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'FF_Human_Ovarian_Adenocarcinoma'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_Prime_Human_Ovary_FF/Xenium_Prime_Human_Ovary_FF_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

File already fully downloaded: /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/FF_Human_Ovarian_Adenocarcinoma/Xenium_Prime_Human_Ovary_FF_outs.zip
Directory already extracted: /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/FF_Human_Ovarian_Adenocarcinoma/Raw_Data. Skipping unzip.
Saved AnnData object to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/FF_Human_Ovarian_Adenocarcinoma/adata.h5ad
AnnData object with n_obs × n_vars = 1157659 × 5001
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


# FFPE Human Skin Primary Dermal Melanoma
https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-skin

In [4]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'Primary_Dermal_Melanoma'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Skin_FFPE/Xenium_Prime_Human_Skin_FFPE_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

Saved to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/Xenium_Prime_Human_Skin_FFPE_outs.zip
Extracting /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/Xenium_Prime_Human_Skin_FFPE_outs.zip to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/Raw_Data...
Extracted to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/Raw_Data
Saved AnnData object to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Primary_Dermal_Melanoma/adata.h5ad
AnnData object with n_obs × n_vars = 112551 × 5006
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


# FFPE Human Prostate Adenocarcinoma
https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-prostate

In [6]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'Human_Prostate_Adenocarcinoma'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://cf.10xgenomics.com/samples/xenium/3.0.0/Xenium_Prime_Human_Prostate_FFPE/Xenium_Prime_Human_Prostate_FFPE_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

Saved to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Prostate_Adenocarcinoma/Xenium_Prime_Human_Prostate_FFPE_outs.zip
Extracting /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Prostate_Adenocarcinoma/Xenium_Prime_Human_Prostate_FFPE_outs.zip to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Prostate_Adenocarcinoma/Raw_Data...
Extracted to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Prostate_Adenocarcinoma/Raw_Data
Saved AnnData object to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Prostate_Adenocarcinoma/adata.h5ad
AnnData object with n_obs × n_vars = 193000 × 5006
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obs

# Post-Xenium Technical Note: Xenium v1 and Xenium Prime 5K for FFPE Human Lung Cancer (Only using V2 5k (experiment 2))
https://www.10xgenomics.com/datasets/xenium-human-lung-cancer-post-xenium-technote

In [8]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'Human_Lung'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_Prime_Human_Lung_Cancer_FFPE/Xenium_Prime_Human_Lung_Cancer_FFPE_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

Saved to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/Xenium_Prime_Human_Lung_Cancer_FFPE_outs.zip
Extracting /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/Xenium_Prime_Human_Lung_Cancer_FFPE_outs.zip to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/Raw_Data...
Extracted to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/Raw_Data
Saved AnnData object to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/adata.h5ad
AnnData object with n_obs × n_vars = 278328 × 5001
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


# FFPE Human Breast Cancer with 5K Human Pan Tissue and Pathways Panel plus 100 Custom Genes
https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-breast-cancer

In [9]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'Human_Breast'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_Prime_Breast_Cancer_FFPE/Xenium_Prime_Breast_Cancer_FFPE_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

Saved to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Breast/Xenium_Prime_Breast_Cancer_FFPE_outs.zip
Extracting /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Breast/Xenium_Prime_Breast_Cancer_FFPE_outs.zip to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Breast/Raw_Data...
Extracted to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Breast/Raw_Data
Saved AnnData object to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Breast/adata.h5ad
AnnData object with n_obs × n_vars = 699110 × 5101
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'


# FFPE Human Cervical Cancer with 5K Human Pan Tissue and Pathways Panel plus 100 Custom Genes
https://www.10xgenomics.com/datasets/xenium-prime-ffpe-human-cervical-cancer

In [10]:
# Idempotent. Will not overwrite if already done.
data_set_name = 'Human_Cervical_Cancer'
this_dir = f'{XENIUM_DATA_DIR}/{data_set_name}'
unzip_to_dir = f'{this_dir}/Raw_Data'
url = "https://s3-us-west-2.amazonaws.com/10x.files/samples/xenium/3.0.0/Xenium_Prime_Cervical_Cancer_FFPE/Xenium_Prime_Cervical_Cancer_FFPE_outs.zip"

zip_path = download_file(url, this_dir)
unzip_file(zip_path, unzip_to_dir)

adata = load_and_save_adata(unzip_to_dir, f'{this_dir}/adata.h5ad', overwrite=False)

Saved to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Cervical_Cancer/Xenium_Prime_Cervical_Cancer_FFPE_outs.zip
Extracting /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Cervical_Cancer/Xenium_Prime_Cervical_Cancer_FFPE_outs.zip to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Cervical_Cancer/Raw_Data...
Extracted to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Cervical_Cancer/Raw_Data
Saved AnnData object to /global/scratch/users/fosterangus/MetabTravLR/Data/Xenium/Human_Cervical_Cancer/adata.h5ad
AnnData object with n_obs × n_vars = 840387 × 5101
    obs: 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method'
    var: 'gene_ids', 'feature_types', 'genome'
    obsm: 'spatial'
